# [2장 2강] 실습: 선형회귀 모델 학습 방법

## 실습 목표

- 회귀 모델의 예측 오차를 MAE와 RMSE로 평가할 수 있다.
- NumPy 최소제곱법으로 선형회귀의 Weight와 Bias를 계산할 수 있다.
- scikit-learn으로 다중 선형회귀 모델을 학습할 수 있다.
- PyTorch와 Gradient Descent로 선형회귀 모델을 반복 학습할 수 있다.
- 세 가지 구현 방법의 파라미터와 성능을 비교할 수 있다.

## 진행 방식

- Google Colab 또는 Jupyter Notebook에서 진행합니다.
- 각 문제의 설명과 요구사항을 확인한 뒤 코드 셀을 작성합니다.
- 실행 결과를 확인하고 문제에 제시된 질문에 답합니다.

## 사용 데이터

- 파일: `insurance(2).csv`
- 입력 Feature: `age`, `sex`, `bmi`, `children`, `smoker`, `region`
- 예측 대상 Label: `charges`

## 실습 준비: 라이브러리 및 데이터 불러오기

Google Colab에서는 파일을 Google Drive의 `MyDrive`에 저장합니다. Jupyter Notebook에서는 노트북과 같은 폴더에 저장합니다.

In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

try:
    from google.colab import drive
    drive.mount('/content/drive')
    file_path = '/content/drive/MyDrive/insurance.csv'
except ModuleNotFoundError:
    file_path = r'C:\Machine_Learning_worksheet\insurance.csv'

insurance_df = pd.read_csv(file_path)
display(insurance_df.head())
print('데이터 크기:', insurance_df.shape)
insurance_df.info()

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


데이터 크기: (1338, 7)
<class 'pandas.DataFrame'>
RangeIndex: 1338 entries, 0 to 1337
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       1338 non-null   int64  
 1   sex       1338 non-null   str    
 2   bmi       1338 non-null   float64
 3   children  1338 non-null   int64  
 4   smoker    1338 non-null   str    
 5   region    1338 non-null   str    
 6   charges   1338 non-null   float64
dtypes: float64(2), int64(2), str(3)
memory usage: 73.3 KB


## 필수 1: 의료비 데이터 확인 및 학습 데이터 준비

### 문제 1-1: 데이터 확인과 전처리 수행하기

#### 문제 설명

보험사는 고객 특성을 이용해 의료비를 예측하려고 합니다. 모델 학습 전에 데이터 품질과 `charges` 분포를 확인하고, 문자형 Feature를 숫자로 변환한 뒤 학습 데이터와 테스트 데이터로 나눕니다.

#### 요구사항

1. 중복 행 수와 전체 결측치 수를 확인합니다.
2. 중복 행을 제거하고 데이터 크기를 출력합니다.
3. `charges`의 평균, 중앙값, 최솟값, 최댓값을 출력합니다.
4. `sex`, `smoker`, `region`에 첫 번째 범주를 제외한 One-Hot Encoding을 적용합니다.
5. 입력 데이터 `X`와 예측 대상 `y`를 분리합니다.
6. `random_state=42`를 사용하여 학습용 80%, 테스트용 20%로 분리합니다.
7. 수치형 입력 컬럼인 `age`, `bmi`, `children`에 StandardScaler를 적용하되, 학습 데이터에는 `fit_transform()`, 테스트 데이터에는 `transform()`을 사용합니다.
8. 학습 데이터와 테스트 데이터의 크기 및 입력 컬럼을 출력합니다.

#### 출력 결과

- 중복 행 수 `1`, 전체 결측치 수 `0`
- 중복 제거 후 데이터 크기 `(1337, 7)`
- `charges` 주요 통계값
- 인코딩된 입력 컬럼 8개
- 학습 데이터 `(1069, 8)`, 테스트 데이터 `(268, 8)`

#### 결과 해석 작성

> 테스트 데이터에 `fit_transform()`이 아니라 `transform()`만 사용하는 이유는 무엇인가요?

In [8]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# (앞 셀에서 insurance_df 를 이미 불러온 상태라고 가정)

# 1. 중복 행 수와 전체 결측치 수 확인
print('중복 행 수:', insurance_df.duplicated().sum(),
      ', 전체 결측치 수:', insurance_df.isnull().sum().sum())

# 2. 중복 행 제거 후 데이터 크기 출력
insurance_df = insurance_df.drop_duplicates().reset_index(drop=True)
print('중복 제거 후 데이터 크기:', insurance_df.shape)

# 3. charges 주요 통계값
print('charges 통계값')
print('평균:', insurance_df['charges'].mean().round(2))
print('중앙값:', insurance_df['charges'].median())
print('최솟값:', insurance_df['charges'].min())
print('최댓값:', insurance_df['charges'].max())

# 4. sex, smoker, region 에 첫 번째 범주 제외 One-Hot Encoding
insurance_encoded = pd.get_dummies(
    insurance_df,
    columns=['sex', 'smoker', 'region'],
    drop_first=True
)

# 5. 입력 X 와 예측 대상 y 분리
X = insurance_encoded.drop(columns='charges')
y = insurance_encoded['charges']
print('인코딩된 입력 컬럼 수:', X.shape[1])

# 6. random_state=42, 학습 80% / 테스트 20% 분리
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 7. age, bmi, children 에 StandardScaler 적용
num_cols = ['age', 'bmi', 'children']
scaler = StandardScaler()
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])

# 8. 학습/테스트 데이터 크기 및 입력 컬럼 출력
print('학습 데이터:', X_train.shape, ', 테스트 데이터:', X_test.shape)
print('입력 컬럼:', list(X_train.columns))

중복 행 수: 0 , 전체 결측치 수: 0
중복 제거 후 데이터 크기: (1337, 7)
charges 통계값
평균: 13279.12
중앙값: 9386.1613
최솟값: 1121.8739
최댓값: 63770.42801
인코딩된 입력 컬럼 수: 8
학습 데이터: (1069, 8) , 테스트 데이터: (268, 8)
입력 컬럼: ['age', 'bmi', 'children', 'sex_male', 'smoker_yes', 'region_northwest', 'region_southeast', 'region_southwest']


## 필수 2: scikit-learn 다중 선형회귀 모델 학습

### 문제 2-1: 다중 선형회귀 모델 학습 및 평가하기

#### 문제 설명

표준화된 여러 입력 Feature를 동시에 사용하는 다중 선형회귀 기준 모델을 학습하고, 테스트 데이터의 의료비를 예측한 뒤 MAE와 RMSE로 평가합니다.

#### 요구사항

1. `LinearRegression` 모델을 생성합니다.
2. 표준화된 학습 데이터로 모델을 학습합니다.
3. 테스트 데이터의 의료비를 예측합니다.
4. Bias와 각 입력 Feature의 Weight를 출력합니다.
5. MAE와 RMSE를 계산합니다.
6. 실제값과 예측값 앞 5개를 비교합니다.
7. MAE와 RMSE의 의미와 RMSE가 더 크게 나타나는 이유를 설명합니다.

#### 결과 해석 작성

> MAE와 RMSE는 각각 무엇을 의미하며, 이 결과에서 RMSE가 MAE보다 큰 이유는 무엇인가요?

In [9]:
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error

# 1. LinearRegression 모델 생성
model = LinearRegression()

# 2. 표준화된 학습 데이터로 학습
model.fit(X_train, y_train)

# 3. 테스트 데이터 의료비 예측
y_pred = model.predict(X_test)

# 4. Bias(절편)와 각 Feature의 Weight(계수) 출력
print('Bias (절편):', model.intercept_)
print('Feature별 Weight (계수):')
for col, coef in zip(X_train.columns, model.coef_):
    print(f'  {col}: {coef}')

# 5. MAE와 RMSE 계산
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print('\nMAE:', mae)
print('RMSE:', rmse)

# 6. 실제값과 예측값 앞 5개 비교
comparison = pd.DataFrame({
    '실제값': y_test.values[:5],
    '예측값': y_pred[:5]
})
print('\n실제값 vs 예측값 (앞 5개)')
print(comparison)

Bias (절편): 8947.952195069174
Feature별 Weight (계수):
  age: 3472.9755534303567
  bmi: 1927.82825100964
  children: 636.501185297569
  sex_male: -101.54205399031055
  smoker_yes: 23077.76459286808
  region_northwest: -391.7614547836059
  region_southeast: -838.9196157269558
  region_southwest: -659.1397515529818

MAE: 4177.045561036324
RMSE: 5956.342894363587

실제값 vs 예측값 (앞 5개)
           실제값           예측값
0   8688.85885   8143.693884
1   5708.86700   5737.115683
2  11436.73815  14369.314876
3  38746.35510  31745.513636
4   4463.20510   8962.386657


## 필수 3: NumPy 최소제곱법으로 선형회귀 직접 구현

### 문제 3-1: 최소제곱법으로 Weight와 Bias 계산하기

#### 문제 설명

NumPy 행렬 연산과 최소제곱법을 이용해 Weight와 Bias를 직접 계산하고 scikit-learn 결과와 비교합니다.

#### 요구사항

1. Bias 계산을 위해 학습 데이터 앞에 값이 모두 `1`인 컬럼을 추가합니다.
2. `np.linalg.pinv()`를 사용하여 최소제곱법의 파라미터를 계산합니다.
3. 첫 번째 파라미터를 Bias, 나머지를 Weight로 구분합니다.
4. 테스트 데이터에도 값이 모두 `1`인 컬럼을 추가하고 의료비를 예측합니다.
5. MAE와 RMSE를 계산합니다.
6. scikit-learn 결과와 Bias, Weight, 성능이 비슷한지 확인합니다.

#### 결과 해석 작성

> NumPy와 scikit-learn의 결과가 거의 같은 이유는 무엇인가요?

In [10]:
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error

# NumPy 배열로 변환
X_train_np = X_train.to_numpy(dtype=float)
X_test_np = X_test.to_numpy(dtype=float)
y_train_np = y_train.to_numpy(dtype=float)
y_test_np = y_test.to_numpy(dtype=float)

# 1. 학습 데이터 앞에 값이 모두 1인 컬럼(bias 항) 추가
X_train_b = np.hstack([np.ones((X_train_np.shape[0], 1)), X_train_np])

# 2. np.linalg.pinv() 로 최소제곱법 파라미터 계산
#    theta = pinv(X_b) @ y
theta = np.linalg.pinv(X_train_b) @ y_train_np

# 3. 첫 번째 파라미터 = Bias, 나머지 = Weight
bias = theta[0]
weights = theta[1:]
print('NumPy Bias:', bias)
print('NumPy Weights:')
for col, w in zip(X_train.columns, weights):
    print(f'  {col}: {w}')

# 4. 테스트 데이터에도 1인 컬럼 추가 후 예측
X_test_b = np.hstack([np.ones((X_test_np.shape[0], 1)), X_test_np])
y_pred_np = X_test_b @ theta

# 5. MAE, RMSE 계산
mae_np = mean_absolute_error(y_test_np, y_pred_np)
rmse_np = np.sqrt(mean_squared_error(y_test_np, y_pred_np))
print('\nNumPy MAE:', mae_np)
print('NumPy RMSE:', rmse_np)

# 6. scikit-learn 결과와 비교
print('\n--- scikit-learn 비교 ---')
print('sklearn Bias:', model.intercept_)
print('Bias 차이:', abs(bias - model.intercept_))
print('Weight 최대 차이:', np.max(np.abs(weights - model.coef_)))

y_pred_sk = model.predict(X_test)
print('sklearn MAE:', mean_absolute_error(y_test, y_pred_sk))
print('sklearn RMSE:', np.sqrt(mean_squared_error(y_test, y_pred_sk)))


NumPy Bias: 8947.952195069163
NumPy Weights:
  age: 3472.975553430358
  bmi: 1927.828251009643
  children: 636.5011852975598
  sex_male: -101.54205399028436
  smoker_yes: 23077.764592868087
  region_northwest: -391.7614547836002
  region_southeast: -838.9196157269595
  region_southwest: -659.1397515529602

NumPy MAE: 4177.045561036326
NumPy RMSE: 5956.342894363585

--- scikit-learn 비교 ---
sklearn Bias: 8947.952195069174
Bias 차이: 1.0913936421275139e-11
Weight 최대 차이: 2.6190605240117293e-11
sklearn MAE: 4177.045561036324
sklearn RMSE: 5956.342894363587


## 필수 4: PyTorch 경사하강법으로 선형회귀 학습

### 문제 4-1: Gradient Descent로 모델 학습하기

#### 문제 설명

PyTorch로 선형회귀 모델을 만들고 SGD를 이용해 MSE Loss가 감소하도록 Weight와 Bias를 반복적으로 업데이트합니다.

#### 요구사항

1. 재현성을 위해 `torch.manual_seed(42)`를 설정합니다.
2. 학습·테스트 입력과 Label을 `float32` Tensor로 변환합니다.
3. `torch.nn.Linear`로 선형회귀 모델을 생성합니다.
4. MSELoss와 SGD Optimizer를 사용하고 Learning Rate를 `0.01`로 설정합니다.
5. 2,000 Epoch 동안 학습하고 첫 Epoch와 200 Epoch마다 Loss를 출력합니다.
6. 학습 전후 Loss를 비교합니다.
7. 테스트 데이터로 MAE와 RMSE를 계산하고 Bias와 Weight를 출력합니다.
8. Gradient Descent가 Loss를 줄이는 원리를 설명합니다.

#### 결과 해석 작성

> Epoch가 증가하면서 Loss가 감소하는 이유는 무엇이며, Gradient Descent는 어떤 방향으로 파라미터를 업데이트하나요?

In [11]:
import torch
import torch.nn as nn
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error

# 1. 재현성을 위한 시드 설정
torch.manual_seed(42)

# 2. 입력과 Label을 float32 Tensor로 변환
X_train_t = torch.tensor(X_train.to_numpy(dtype='float32'))
X_test_t = torch.tensor(X_test.to_numpy(dtype='float32'))
# y는 (N,) -> (N,1) 형태로 맞춰줌 (모델 출력과 shape 일치)
y_train_t = torch.tensor(y_train.to_numpy(dtype='float32')).view(-1, 1)
y_test_t = torch.tensor(y_test.to_numpy(dtype='float32')).view(-1, 1)

# 3. nn.Linear 로 선형회귀 모델 생성
n_features = X_train_t.shape[1]
model_torch = nn.Linear(n_features, 1)

# 4. MSELoss, SGD Optimizer, Learning Rate = 0.01
criterion = nn.MSELoss()
optimizer = torch.optim.SGD(model_torch.parameters(), lr=0.01)

# 5. 2000 Epoch 학습, 첫 Epoch과 200 Epoch마다 Loss 출력
n_epochs = 2000

# 학습 전 Loss 기록 (6번 비교용)
model_torch.eval()
with torch.no_grad():
    loss_before = criterion(model_torch(X_train_t), y_train_t).item()

model_torch.train()
for epoch in range(1, n_epochs + 1):
    # 순전파
    y_pred_train = model_torch(X_train_t)
    loss = criterion(y_pred_train, y_train_t)

    # 역전파 및 파라미터 업데이트
    optimizer.zero_grad()   # 이전 gradient 초기화
    loss.backward()         # gradient 계산
    optimizer.step()        # 파라미터 업데이트

    if epoch == 1 or epoch % 200 == 0:
        print(f'Epoch {epoch:4d} | Loss: {loss.item():.4f}')

# 6. 학습 전후 Loss 비교
model_torch.eval()
with torch.no_grad():
    loss_after = criterion(model_torch(X_train_t), y_train_t).item()
print(f'\n학습 전 Loss: {loss_before:.4f}')
print(f'학습 후 Loss: {loss_after:.4f}')

# 7. 테스트 데이터로 MAE, RMSE 계산 및 Bias, Weight 출력
with torch.no_grad():
    y_pred_test = model_torch(X_test_t)

y_pred_np = y_pred_test.numpy().flatten()
y_test_np = y_test_t.numpy().flatten()

mae = mean_absolute_error(y_test_np, y_pred_np)
rmse = np.sqrt(mean_squared_error(y_test_np, y_pred_np))
print(f'\nMAE: {mae:.4f}')
print(f'RMSE: {rmse:.4f}')

print('\nBias:', model_torch.bias.item())
print('Weights:')
for col, w in zip(X_train.columns, model_torch.weight.detach().numpy().flatten()):
    print(f'  {col}: {w}')
    

Epoch    1 | Loss: 306685920.0000
Epoch  200 | Loss: 59295072.0000
Epoch  400 | Loss: 43728264.0000
Epoch  600 | Loss: 39126816.0000
Epoch  800 | Loss: 37722288.0000
Epoch 1000 | Loss: 37272484.0000
Epoch 1200 | Loss: 37115904.0000
Epoch 1400 | Loss: 37053728.0000
Epoch 1600 | Loss: 37024596.0000
Epoch 1800 | Loss: 37008692.0000
Epoch 2000 | Loss: 36999028.0000

학습 전 Loss: 306685920.0000
학습 후 Loss: 36998988.0000

MAE: 4180.6958
RMSE: 5969.9487

Bias: 8671.2275390625
Weights:
  age: 3477.416748046875
  bmi: 1901.631591796875
  children: 637.314453125
  sex_male: -58.84477615356445
  smoker_yes: 23051.2734375
  region_northwest: -79.1548080444336
  region_southeast: -501.85284423828125
  region_southwest: -337.4072570800781


## 심화 1: 세 가지 선형회귀 모델 비교

### 문제 5-1: 모델별 파라미터와 성능 비교하기

#### 문제 설명

동일한 데이터로 학습한 NumPy, scikit-learn, PyTorch 모델의 Bias, Weight와 성능을 표로 비교합니다.

#### 요구사항

1. 세 모델의 Bias, MAE, RMSE를 하나의 표로 만듭니다.
2. 각 입력 Feature의 Weight를 모델별로 비교합니다.
3. 세 모델의 결과가 비슷한지 확인합니다.
4. 계산 방법이 다른데도 결과가 비슷한 이유를 설명합니다.

#### 결과 해석 작성

> 최소제곱법과 Gradient Descent의 계산 과정이 다른데도 세 모델의 결과가 비슷하게 나타나는 이유는 무엇인가요?

In [ ]:
# Generalization(일반화) : 학습 때 보지 못한 새로운 데이터에서도 비슷한 결과를 냄
# -> 연습문제(Train)뿐만 아니라 오늘 처음보는 시험(Test) 데이터에서도 비슷한 결과를 내는 것

# overfitting(과대 적합) : Train 데이터에서는 성능이 좋지만 새로운 데이터에서는 성능이 떨어지는 현상
#

#regularization(정규화) : loss에 가중치(weight)를 추가하여 모델이 과대적합되지 않도록 하는 방법


#Ridge, L2 : Weight의 제곱합을 벌점으로 사용, Weight를 최소화 하려고 만듦(0은 아님)
#-> 여러 사람에게 부담을 나누어 문제의 무게를 조금씩 줄인다

#Lasso, L1 : Weight의 절댓값 합을 벌점으로 사용, Weight를 0으로 만들려고 만듦
#-> 여러 사람에게 부담을 나누지 않고 한사람에게 부담을 몰아주어 문제의 무게를 줄인다


alpha =  정규화 벌점의 강도를 조절하는 하이퍼파라미터
-> 교통 단속 벌금이 클수록 과속을 더 강하게 억제한다
alph = 0에 가까우면 약한 정규화, alpha가 커질수록 강한 정규화

# 하이퍼 파라미터 : 모델에게 학습 전 사람이 입력하는 설정값
# -> 모델이 학습하면서 최적화하는 파라미터(Weight, Bias)와 달리 사람이 직접 설정하는 값

## 실습 마무리

1. 최소제곱법과 Gradient Descent의 가장 큰 차이는 무엇인가요?
2. Learning Rate가 너무 크거나 너무 작으면 어떤 문제가 발생하나요?
3. 세 모델을 공정하게 비교하려면 어떤 조건을 같게 유지해야 하나요?